# Notebook 11: Cross-Cohort Validation — GSE64018 (Gupta et al. 2014)

**Validation Dataset:** [GSE64018](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE64018) — Gupta et al. 2014, *Nature Communications*  
**Title:** "Transcriptome analysis reveals dysregulation of innate immune response genes and neuronal activity-dependent genes in autism"  
**Platform:** RNA-seq (Illumina HiSeq 2000)  
**Samples:** Postmortem brain samples (ASD + controls) across brain regions  

**Discovery Dataset (Notebook 10):** [GSE28521](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE28521) — Voineagu et al. 2011, *Nature* (microarray)  

**Framework:** [pathway-subtyping](https://pypi.org/project/pathway-subtyping/) v0.3.0  
**Author:** Rohit Chauhan ([ORCID: 0009-0003-9895-4629](https://orcid.org/0009-0003-9895-4629))  
**Zenodo DOI:** [10.5281/zenodo.18442426](https://doi.org/10.5281/zenodo.18442426)  

---

## What this notebook does

1. Downloads GSE64018 RNA-seq expression data from GEO
2. Preprocesses the expression matrix (gene-level, log2-transform)
3. Applies the same 15 autism pathways and ssGSEA scoring as Notebook 10
4. **Independent discovery:** Runs pathway-based GMM subtyping on frontal cortex
5. **Cross-cohort projection:** Projects GSE64018 samples into the GSE28521 model
6. Tests whether the GABA-Collapsed subtype replicates across platforms
7. Generates cross-validation figures for the manuscript

**Why GSE64018?** This is an RNA-seq dataset (vs. microarray in GSE28521), from an independent cohort. Cross-platform replication is the strongest form of validation.

**Runtime:** ~5-10 minutes on Colab Pro

## 1. Setup & Installation

In [ ]:
# Install pathway-subtyping framework with visualization extras
!pip install -q pathway-subtyping[viz]==0.3.0 GEOparse mygene

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
import urllib.request

# Framework imports
from pathway_subtyping import (
    score_pathways_from_expression,
    ExpressionScoringMethod,
    run_clustering,
    ClusteringAlgorithm,
    select_n_clusters,
    ValidationGates,
    characterize_subtypes,
    generate_subtype_heatmap,
    generate_gene_heatmap,
    export_characterization,
    run_benchmark_comparison,
    compute_dim_reduction,
    DimReductionMethod,
)

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Output directories
DATA_DIR = "./data"
OUTPUT_DIR = "./outputs/gse64018"
FC_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "frontal_cortex")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FC_OUTPUT_DIR, exist_ok=True)

print("Setup complete.")

## 2. Download GSE64018 from GEO

GSE64018 is an RNA-seq dataset. GEO series matrix files for RNA-seq sometimes
contain processed expression values (RPKM/FPKM). We download and inspect.

In [ ]:
import GEOparse

print("Downloading GSE64018 from GEO (this may take a minute)...")
gse = GEOparse.get_GEO(geo="GSE64018", destdir=DATA_DIR, silent=True)
print(f"Downloaded. Platform(s): {list(gse.gpls.keys())}")
print(f"Number of samples (GSMs): {len(gse.gsms)}")

# Inspect first sample to understand data structure
first_gsm = list(gse.gsms.values())[0]
print(f"\nFirst sample: {list(gse.gsms.keys())[0]}")
print(f"Title: {first_gsm.metadata.get('title', ['?'])[0]}")
print(f"Source: {first_gsm.metadata.get('source_name_ch1', ['?'])[0]}")
print(f"Characteristics: {first_gsm.metadata.get('characteristics_ch1', [])}")
print(f"\nTable columns: {list(first_gsm.table.columns)}")
print(f"Table shape: {first_gsm.table.shape}")
print(f"\nFirst 5 rows of data:")
print(first_gsm.table.head())

## 3. Extract Sample Metadata

Parse sample characteristics to extract diagnosis (ASD vs Control) and brain region.

In [ ]:
# Extract metadata from all samples
metadata_rows = []
for gsm_name, gsm in gse.gsms.items():
    chars = gsm.metadata.get("characteristics_ch1", [])
    char_dict = {}
    for c in chars:
        if ":" in c:
            key, val = c.split(":", 1)
            char_dict[key.strip().lower()] = val.strip()
    
    title = gsm.metadata.get("title", [""])[0]
    source = gsm.metadata.get("source_name_ch1", [""])[0]
    
    metadata_rows.append({
        "sample_id": gsm_name,
        "title": title,
        "source": source,
        **char_dict,
    })

metadata = pd.DataFrame(metadata_rows).set_index("sample_id")

# Display all metadata columns to understand the structure
print("Metadata columns:", list(metadata.columns))
print(f"\nAll metadata (first 10 rows):")
print(metadata.head(10).to_string())
print(f"\nUnique values per column:")
for col in metadata.columns:
    uniq = metadata[col].unique()
    if len(uniq) <= 15:
        print(f"  {col}: {list(uniq)}")
    else:
        print(f"  {col}: {len(uniq)} unique values")

In [ ]:
# Parse diagnosis and brain region
# GSE64018 metadata structure may vary — adapt parsing based on the output above

def parse_gse64018_metadata(meta_df):
    """Parse diagnosis and brain region from GSE64018 sample metadata.
    
    This function handles multiple possible metadata formats.
    Adjust the column names based on the actual metadata structure.
    """
    result = meta_df.copy()
    
    # --- Diagnosis ---
    # Try common column names for diagnosis/disease status
    dx_col = None
    for col in ['disease state', 'disease status', 'diagnosis', 'condition',
                'disease', 'phenotype', 'group', 'status']:
        if col in result.columns:
            dx_col = col
            break
    
    if dx_col:
        dx_vals = result[dx_col].str.lower()
        result['diagnosis'] = dx_vals.map(
            lambda x: 'ASD' if any(k in str(x) for k in ['autism', 'asd', 'case', 'affected'])
            else 'Control' if any(k in str(x) for k in ['control', 'normal', 'unaffected', 'healthy'])
            else 'Unknown'
        )
    else:
        # Fallback: parse from title or source
        result['diagnosis'] = result['title'].apply(
            lambda x: 'ASD' if any(k in str(x).lower() for k in ['autism', 'asd', 'case'])
            else 'Control' if any(k in str(x).lower() for k in ['control', 'normal'])
            else 'Unknown'
        )
    
    # --- Brain Region ---
    region_col = None
    for col in ['tissue', 'brain region', 'region', 'brain_region',
                'tissue type', 'cell type', 'organ']:
        if col in result.columns:
            region_col = col
            break
    
    if region_col:
        region_vals = result[region_col].str.lower()
        result['brain_region'] = region_vals.map(
            lambda x: 'Frontal_Cortex' if any(k in str(x) for k in ['frontal', 'prefrontal', 'ba9', 'pfc'])
            else 'Temporal_Cortex' if any(k in str(x) for k in ['temporal', 'ba41', 'ba42'])
            else 'Cerebellum' if any(k in str(x) for k in ['cerebellum', 'cerebell'])
            else 'Other_Cortex' if any(k in str(x) for k in ['cortex', 'cortical'])
            else str(x).replace(' ', '_')
        )
    else:
        # Fallback: parse from title or source
        result['brain_region'] = result.apply(
            lambda row: next(
                ('Frontal_Cortex' if 'frontal' in t or 'prefrontal' in t else
                 'Temporal_Cortex' if 'temporal' in t else
                 'Cerebellum' if 'cerebel' in t else
                 'Cortex' if 'cortex' in t or 'cortical' in t else 'Unknown')
                for t in [str(row.get('title', '')).lower() + ' ' + str(row.get('source', '')).lower()]
            ), axis=1
        )
    
    return result

metadata = parse_gse64018_metadata(metadata)

print("\n--- Sample Breakdown ---")
print(metadata.groupby(["brain_region", "diagnosis"]).size().unstack(fill_value=0))
print(f"\nTotal samples: {len(metadata)}")
print(f"ASD: {(metadata['diagnosis'] == 'ASD').sum()}")
print(f"Control: {(metadata['diagnosis'] == 'Control').sum()}")
print(f"Unknown: {(metadata['diagnosis'] == 'Unknown').sum()}")

## 4. Build Expression Matrix

GSE64018 is RNA-seq — the expression data is stored in **supplementary files** (FPKM values),
not in the standard series matrix that GEOparse reads for microarray datasets.

We download the processed FPKM matrix directly from the GEO FTP server.

**Note:** All 24 GSE64018 samples are from temporal cortex (BA41-42-22), not frontal cortex.
This tests cross-region generalization of the GABA-Collapsed subtype found in GSE28521 frontal cortex (BA9).

In [ ]:
# Download the supplementary FPKM file directly from GEO FTP
# GEOparse's pivot_samples() hardcodes 'ID_REF' which doesn't exist for RNA-seq datasets

FPKM_URL = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE64nnn/GSE64018/suppl/GSE64018_adjfpkm_12asd_12ctl.txt.gz"
FPKM_PATH = os.path.join(DATA_DIR, "GSE64018_adjfpkm_12asd_12ctl.txt.gz")

if not os.path.exists(FPKM_PATH):
    print("Downloading GSE64018 adjusted FPKM matrix from GEO FTP...")
    urllib.request.urlretrieve(FPKM_URL, FPKM_PATH)
    print(f"Downloaded: {FPKM_PATH}")
else:
    print(f"Using cached: {FPKM_PATH}")

# Read the FPKM matrix
expression_df = pd.read_csv(FPKM_PATH, sep="\t", index_col=0, compression="gzip")
print(f"\nRaw FPKM matrix: {expression_df.shape[0]} genes x {expression_df.shape[1]} samples")
print(f"Columns: {list(expression_df.columns[:5])}{'...' if len(expression_df.columns) > 5 else ''}")
print(f"Index (first 5): {list(expression_df.index[:5])}")
print(f"\nFirst 3 rows:")
print(expression_df.head(3))

In [ ]:
# Map Ensembl gene IDs to gene symbols and align sample columns to GSM IDs

# --- Step 1: Map columns (brain bank IDs) to GSM IDs ---
# FPKM columns are like "AN02987_ba41.42.22_8.6"
# GSM titles are like "ASD_AN02987_ba41-42-22_8.6"
print("--- Mapping FPKM columns to GSM IDs ---")
print(f"FPKM columns (first 3): {list(expression_df.columns[:3])}")

# Build mapping: extract brain bank ID from both FPKM columns and GSM titles
col_to_gsm = {}
for col in expression_df.columns:
    # Extract brain bank ID (e.g., "AN02987" from "AN02987_ba41.42.22_8.6")
    bank_id = col.split("_")[0]
    
    for gsm_name, gsm in gse.gsms.items():
        title = gsm.metadata.get("title", [""])[0]
        # GSM title format: "ASD_AN02987_ba41-42-22_8.6"
        if bank_id in title:
            col_to_gsm[col] = gsm_name
            break

print(f"Mapped {len(col_to_gsm)}/{len(expression_df.columns)} columns to GSM IDs")
if len(col_to_gsm) > 0:
    expression_df = expression_df.rename(columns=col_to_gsm)
    unmapped = [c for c in expression_df.columns if not c.startswith("GSM")]
    if unmapped:
        print(f"Unmapped columns (dropping): {unmapped}")
        expression_df = expression_df[[c for c in expression_df.columns if c.startswith("GSM")]]
else:
    print("WARNING: Could not map columns. Using positional alignment later.")

# --- Step 2: Convert Ensembl gene IDs to gene symbols ---
print(f"\n--- Ensembl to Gene Symbol Mapping ---")
print(f"Row IDs (first 5): {list(expression_df.index[:5])}")

# Install and use mygene for Ensembl-to-symbol mapping
try:
    import mygene
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "-q", "mygene"])
    import mygene

mg = mygene.MyGeneInfo()

# Get all Ensembl IDs
ensembl_ids = list(expression_df.index)
print(f"Total Ensembl IDs: {len(ensembl_ids)}")

# Query in batches
print("Querying mygene for gene symbol mapping (this may take a moment)...")
results = mg.querymany(ensembl_ids, scopes="ensembl.gene", fields="symbol",
                       species="human", returnall=True, verbose=False)

# Build mapping
ensembl_to_symbol = {}
for hit in results["out"]:
    if "symbol" in hit and "query" in hit:
        ensembl_to_symbol[hit["query"]] = hit["symbol"]

n_mapped = len(ensembl_to_symbol)
print(f"Mapped {n_mapped}/{len(ensembl_ids)} Ensembl IDs to gene symbols ({n_mapped/len(ensembl_ids)*100:.1f}%)")

# Apply mapping
mapped_mask = expression_df.index.isin(ensembl_to_symbol.keys())
expression_df = expression_df.loc[mapped_mask]
expression_df.index = [ensembl_to_symbol[eid] for eid in expression_df.index]

# Collapse duplicate gene symbols (take mean)
expression_df = expression_df.groupby(expression_df.index).mean()
print(f"Gene-level expression: {expression_df.shape[0]} unique genes x {expression_df.shape[1]} samples")

# Transpose to samples x genes
gene_expression = expression_df.T

# Check if data is already log-transformed (the R script applies log2(x+1) then adjusts)
max_val = gene_expression.max().max()
print(f"\nMax expression value: {max_val:.2f}")
if max_val > 30:
    print("Applying log2(x+1) transform")
    gene_expression = np.log2(gene_expression + 1)
else:
    print("Data appears already log-transformed (adjusted FPKM) — no additional transform")

print(f"\nFinal: {gene_expression.shape[0]} samples x {gene_expression.shape[1]} genes")
print(f"Expression range: [{gene_expression.min().min():.2f}, {gene_expression.max().max():.2f}]")

In [ ]:
# Basic QC
print("--- Expression Matrix QC ---")
print(f"Shape: {gene_expression.shape}")
print(f"Missing values: {gene_expression.isna().sum().sum()}")
print(f"Expression range: [{gene_expression.min().min():.2f}, {gene_expression.max().max():.2f}]")
print(f"Mean expression: {gene_expression.mean().mean():.2f}")

# Drop zero-variance genes
gene_var = gene_expression.var()
n_zero_var = (gene_var == 0).sum()
if n_zero_var > 0:
    gene_expression = gene_expression.loc[:, gene_var > 0]
    print(f"Dropped {n_zero_var} zero-variance genes. Remaining: {gene_expression.shape[1]}")

# Fill NaN with column median
if gene_expression.isna().any().any():
    gene_expression = gene_expression.fillna(gene_expression.median())
    print("Filled NaN values with column medians.")

# Align sample IDs between expression and metadata
# If expression columns were mapped to GSM IDs, they'll match metadata index
# If not, we need to build a mapping
expr_samples = set(gene_expression.index)
meta_samples = set(metadata.index)
common_samples = gene_expression.index.intersection(metadata.index)

if len(common_samples) > 0:
    print(f"\nDirect sample alignment: {len(common_samples)} samples matched")
    gene_expression = gene_expression.loc[common_samples]
    metadata = metadata.loc[common_samples]
else:
    print(f"\nNo direct match between expression ({list(expr_samples)[:3]}) and metadata ({list(meta_samples)[:3]})")
    print("Building alignment via sample titles...")
    
    # Map expression sample names to metadata GSM IDs
    expr_to_gsm = {}
    for gsm_name, gsm in gse.gsms.items():
        title = gsm.metadata.get("title", [""])[0]
        # Check if any expression sample name matches this GSM's title
        for expr_name in gene_expression.index:
            if (str(expr_name).lower() in title.lower() or 
                title.lower() in str(expr_name).lower() or
                str(expr_name) == title):
                expr_to_gsm[expr_name] = gsm_name
                break
    
    if expr_to_gsm:
        print(f"Mapped {len(expr_to_gsm)} expression samples to GSM IDs")
        gene_expression = gene_expression.rename(index=expr_to_gsm)
        common_samples = gene_expression.index.intersection(metadata.index)
        gene_expression = gene_expression.loc[common_samples]
        metadata = metadata.loc[common_samples]
    else:
        # Last resort: align by position if same count
        if len(gene_expression) == len(metadata):
            print("Same sample count — aligning by position")
            gene_expression.index = metadata.index
        else:
            print(f"WARNING: Cannot align. Expression has {len(gene_expression)} samples, metadata has {len(metadata)}")

print(f"\nFinal expression matrix: {gene_expression.shape[0]} samples x {gene_expression.shape[1]} genes")
print(f"\n--- Sample Breakdown (aligned) ---")
if "brain_region" in metadata.columns and "diagnosis" in metadata.columns:
    print(metadata.groupby(["brain_region", "diagnosis"]).size().unstack(fill_value=0))
elif "diagnosis" in metadata.columns:
    print(metadata["diagnosis"].value_counts())

## 5. Load Autism Pathways & Score

Apply the same 15 SFARI-derived autism pathways and ssGSEA scoring as Notebook 10.

In [ ]:
# Download the same GMT file used in notebook 10
GMT_URL = "https://raw.githubusercontent.com/topmist-admin/pathway-subtyping-framework/main/data/pathways/autism_pathways.gmt"
GMT_PATH = os.path.join(DATA_DIR, "autism_pathways.gmt")

if not os.path.exists(GMT_PATH):
    urllib.request.urlretrieve(GMT_URL, GMT_PATH)
    print(f"Downloaded autism_pathways.gmt")

# Parse GMT file
pathways = {}
with open(GMT_PATH) as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        parts = line.split("\t")
        if len(parts) >= 3:
            pathways[parts[0]] = parts[2:]

print(f"Loaded {len(pathways)} pathways:")
total_genes = set()
for name, genes in pathways.items():
    available = len(set(genes) & set(gene_expression.columns))
    total_genes.update(genes)
    print(f"  {name}: {len(genes)} genes ({available} found in expression data)")

overlap = len(total_genes & set(gene_expression.columns))
print(f"\nTotal unique pathway genes: {len(total_genes)}")
print(f"Found in GSE64018 expression data: {overlap} ({overlap/len(total_genes)*100:.1f}%)")

In [ ]:
# Score pathways using ssGSEA (same method as notebook 10)
scoring_result = score_pathways_from_expression(
    gene_expression=gene_expression,
    pathways=pathways,
    method=ExpressionScoringMethod.SSGSEA,
    min_genes_per_pathway=2,
    seed=SEED,
    show_progress=True,
)

pathway_scores = scoring_result.pathway_scores

print("\n--- Scoring Report ---")
print(scoring_result.format_report())
print(f"\nPathway score matrix: {pathway_scores.shape}")
print(f"Pathways scored: {scoring_result.n_pathways_scored}")
print(f"Pathways skipped: {scoring_result.n_pathways_skipped}")

## 6. Independent Discovery: Cortex Analysis

GSE64018 samples are all from temporal cortex (BA41-42-22), while the discovery
finding in GSE28521 was in frontal cortex (BA9). We use all 24 cortex samples
for independent subtyping — if the GABA-Collapsed subtype appears in temporal
cortex too, that's even stronger evidence of a brain-wide signature.

In [ ]:
# GSE64018: All samples are temporal cortex (BA41-42-22)
# No need to subset by region — use all samples for discovery

print("Available brain regions:")
if "brain_region" in metadata.columns:
    print(metadata["brain_region"].value_counts())
else:
    print("  No brain_region column — all samples are cortex (BA41-42-22)")

# Use all samples (they're all temporal cortex)
target_region = "Temporal_Cortex (BA41-42-22)"
fc_scores = pathway_scores.copy()
fc_expression = gene_expression.copy()
fc_meta = metadata.copy()

print(f"\nUsing all samples as cortex (region: {target_region})")
print(f"\n--- Cortex Dataset ---")
print(f"Samples: {len(fc_scores)}")
print(f"  ASD:     {(fc_meta['diagnosis'] == 'ASD').sum()}")
print(f"  Control: {(fc_meta['diagnosis'] == 'Control').sum()}")
print(f"Pathways: {fc_scores.shape[1]}")

In [ ]:
# Sweep k=2,3,4 with full validation (same as notebook 10, Section 15b)
fc_all_results = {}

for k in [2, 3, 4]:
    print(f"\n{'='*60}")
    print(f"GSE64018 FRONTAL CORTEX — k={k}")
    print(f"{'='*60}")
    
    # Cluster
    fc_clustering = run_clustering(
        data=fc_scores.values,
        n_clusters=k,
        algorithm=ClusteringAlgorithm.GMM,
        seed=SEED,
    )
    fc_labels = fc_clustering.labels
    
    print(f"\nSilhouette: {fc_clustering.silhouette:.4f}")
    
    # Cross-tab with diagnosis
    fc_subtype_meta = fc_meta.copy()
    fc_subtype_meta["subtype"] = fc_labels
    ct = pd.crosstab(fc_subtype_meta["subtype"], fc_subtype_meta["diagnosis"], margins=True)
    print(f"\nSubtype x Diagnosis:")
    print(ct)
    
    # Validation gates
    fc_gates = ValidationGates(
        seed=SEED,
        n_permutations=200,
        n_bootstrap=100,
        stability_threshold=0.8,
        null_ari_max=0.15,
        show_progress=False,
    )
    
    fc_val = fc_gates.run_all(
        pathway_scores=fc_scores,
        cluster_labels=fc_labels,
        pathways=pathways,
        gene_burdens=fc_expression,
        n_clusters=k,
        gmm_seed=SEED,
    )
    
    print(f"\nValidation Gates:")
    n_passed = 0
    for gate in fc_val.results:
        status = "PASS" if gate.passed else "FAIL"
        if gate.passed:
            n_passed += 1
        print(f"  [{status}] {gate.name}: {gate.metric_name} = {gate.metric_value:.4f}")
    
    # Characterize
    fc_char = characterize_subtypes(
        pathway_scores=fc_scores,
        cluster_labels=fc_labels,
        gene_burdens=fc_expression,
        pathways=pathways,
        fdr_alpha=0.05,
        top_n_genes=15,
        seed=SEED,
    )
    
    fc_all_results[k] = {
        "clustering": fc_clustering,
        "labels": fc_labels,
        "validation": fc_val,
        "characterization": fc_char,
        "n_gates_passed": n_passed,
        "silhouette": fc_clustering.silhouette,
        "meta": fc_subtype_meta,
    }

In [ ]:
# Select best k and visualize
print("=" * 60)
print("GSE64018 FRONTAL CORTEX: k COMPARISON")
print("=" * 60)
print(f"\n{'k':<4} {'Silhouette':<12} {'Gates Passed':<14}")
print("-" * 30)
for k in [2, 3, 4]:
    r = fc_all_results[k]
    print(f"{k:<4} {r['silhouette']:<12.4f} {r['n_gates_passed']}/{len(r['validation'].results)}")

# Select best k: prioritize gates passed, then silhouette
best_k = max(fc_all_results.keys(),
             key=lambda k: (fc_all_results[k]["n_gates_passed"],
                            fc_all_results[k]["silhouette"]))

print(f"\nBest k: {best_k} (silhouette={fc_all_results[best_k]['silhouette']:.4f})")

best_r = fc_all_results[best_k]
best_labels = best_r["labels"]
best_char = best_r["characterization"]
best_meta = best_r["meta"]

# Print characterization
print("\n" + best_char.format_report())

# Check: is there a subtype with only ASD samples?
ct = pd.crosstab(best_meta["subtype"], best_meta["diagnosis"])
print("\n--- KEY QUESTION: Is there an ASD-only subtype? ---")
for subtype in ct.index:
    n_asd = ct.loc[subtype, "ASD"] if "ASD" in ct.columns else 0
    n_ctl = ct.loc[subtype, "Control"] if "Control" in ct.columns else 0
    purity = n_asd / (n_asd + n_ctl) * 100 if (n_asd + n_ctl) > 0 else 0
    print(f"  Subtype {subtype}: {n_asd} ASD + {n_ctl} Control = {purity:.0f}% ASD")

In [ ]:
# Visualize: PCA scatter, pathway heatmap, gene heatmap

# PCA scatter
fc_embedding, fc_pca_meta = compute_dim_reduction(
    pathway_scores=fc_scores,
    method=DimReductionMethod.PCA,
    n_components=2,
    seed=SEED,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: by subtype
colors_k = plt.cm.Set1(np.linspace(0, 1, best_k))
for i in range(best_k):
    mask = best_labels == i
    n_asd = (best_meta.loc[fc_scores.index[mask], "diagnosis"] == "ASD").sum()
    n_ctl = mask.sum() - n_asd
    axes[0].scatter(fc_embedding[mask, 0], fc_embedding[mask, 1], c=[colors_k[i]],
                    label=f"Subtype {i} (n={mask.sum()}: {n_asd}A/{n_ctl}C)",
                    s=80, alpha=0.8, edgecolors="k", linewidth=0.5)
axes[0].set_xlabel(f"PC1 ({fc_pca_meta['explained_variance_ratio'][0]*100:.1f}%)")
axes[0].set_ylabel(f"PC2 ({fc_pca_meta['explained_variance_ratio'][1]*100:.1f}%)")
axes[0].set_title(f"GSE64018 Cortex Subtypes (k={best_k})")
axes[0].legend(fontsize=9)

# Right: by diagnosis
for dx, color in [("ASD", "coral"), ("Control", "steelblue")]:
    mask = fc_meta.loc[fc_scores.index, "diagnosis"].values == dx
    axes[1].scatter(fc_embedding[mask, 0], fc_embedding[mask, 1], c=color,
                    label=dx, s=80, alpha=0.8, edgecolors="k", linewidth=0.5)
axes[1].set_xlabel(f"PC1 ({fc_pca_meta['explained_variance_ratio'][0]*100:.1f}%)")
axes[1].set_ylabel(f"PC2 ({fc_pca_meta['explained_variance_ratio'][1]*100:.1f}%)")
axes[1].set_title("GSE64018 Cortex — Diagnosis")
axes[1].legend(fontsize=9)

plt.suptitle(f"GSE64018 (Gupta et al. 2014): Independent Discovery", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(FC_OUTPUT_DIR, "gse64018_fc_pca_scatter.png"), dpi=150, bbox_inches="tight")
plt.show()

# Pathway heatmap
fig_hm = generate_subtype_heatmap(
    best_char,
    output_path=os.path.join(FC_OUTPUT_DIR, "gse64018_fc_subtype_heatmap.png"),
    figsize=(14, max(4, best_k * 1.5)),
)
plt.show()

# Gene heatmap
fig_genes = generate_gene_heatmap(
    best_char,
    output_path=os.path.join(FC_OUTPUT_DIR, "gse64018_fc_gene_heatmap.png"),
    figsize=(16, max(5, best_k * 2)),
    top_n=15,
)
plt.show()

In [ ]:
# Benchmark on GSE64018 cortex
print("Running benchmark comparison (GSE64018 cortex)...")
fc_bench = run_benchmark_comparison(
    gene_burdens=fc_expression,
    pathway_scores=fc_scores,
    pathways=pathways,
    n_clusters=best_k,
    seed=SEED,
)
print("\n" + fc_bench.format_report())

# Visualize benchmark
methods = list(fc_bench.method_results.keys())
sils = [fc_bench.method_results[m].silhouette for m in methods]

fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#2ecc71" if m == fc_bench.best_method else "#3498db" for m in methods]
bars = ax.barh(methods, sils, color=colors)
ax.set_xlabel("Silhouette Score")
ax.set_title(f"GSE64018 Cortex Benchmark (k={best_k})")
for bar, val in zip(bars, sils):
    ax.text(max(bar.get_width() + 0.005, 0.01), bar.get_y() + bar.get_height()/2,
            f"{val:.3f}", va="center", fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(FC_OUTPUT_DIR, "gse64018_fc_benchmark.png"), dpi=150, bbox_inches="tight")
plt.show()

## 7. Cross-Cohort Projection

Train a GMM on the GSE28521 frontal cortex pathway scores (discovery cohort),
then project GSE64018 cortex samples into the same feature space and predict
subtype labels. This tests whether the GABA-Collapsed subtype generalizes.

In [ ]:
# Step 1: Rebuild GSE28521 discovery model
# We need the GSE28521 frontal cortex pathway scores to train the GMM.
# Option A: Load from notebook 10 outputs (if available)
# Option B: Re-run GSE28521 analysis in this notebook

# Try Option A first
gse28521_fc_path = "./outputs/gse28521/frontal_cortex/fc_pathway_scores.csv"
gse28521_meta_path = "./outputs/gse28521/frontal_cortex/fc_sample_metadata_with_subtypes.csv"

if os.path.exists(gse28521_fc_path) and os.path.exists(gse28521_meta_path):
    print("Loading GSE28521 frontal cortex results from notebook 10 outputs...")
    discovery_scores = pd.read_csv(gse28521_fc_path, index_col=0)
    discovery_meta = pd.read_csv(gse28521_meta_path, index_col=0)
    discovery_labels = discovery_meta["subtype"].values
    print(f"Loaded: {discovery_scores.shape[0]} samples x {discovery_scores.shape[1]} pathways")
    print(f"Discovery subtypes: {pd.Series(discovery_labels).value_counts().to_dict()}")
else:
    print("GSE28521 outputs not found. Re-running discovery analysis...")
    
    # Option B: Quick re-run of GSE28521 frontal cortex
    print("\nDownloading and processing GSE28521 for cross-cohort validation...")
    gse_disc = GEOparse.get_GEO(geo="GSE28521", destdir=DATA_DIR, silent=True)
    
    # Quick metadata extraction
    disc_meta_rows = []
    for gsm_name, gsm in gse_disc.gsms.items():
        title = gsm.metadata.get("title", [""])[0]
        parts = title.split("_")
        diagnosis = "ASD" if parts[0] == "A" else "Control" if parts[0] == "C" else "Unknown"
        region_map = {"C": "Cerebellum", "F": "Frontal_Cortex", "T": "Temporal_Cortex"}
        region = region_map.get(parts[-1], "Unknown") if len(parts) >= 3 else "Unknown"
        disc_meta_rows.append({"sample_id": gsm_name, "diagnosis": diagnosis, "brain_region": region})
    disc_meta = pd.DataFrame(disc_meta_rows).set_index("sample_id")
    
    # Expression matrix
    disc_expr_df = gse_disc.pivot_samples("VALUE").apply(pd.to_numeric, errors="coerce").dropna(how="all")
    disc_gpl = list(gse_disc.gpls.values())[0]
    disc_sym_col = None
    for col in ["Symbol", "Gene Symbol", "GENE_SYMBOL", "Gene_Symbol"]:
        if col in disc_gpl.table.columns:
            disc_sym_col = col
            break
    if disc_sym_col is None:
        for col in disc_gpl.table.columns:
            if 'symbol' in col.lower():
                disc_sym_col = col
                break
    disc_p2g = disc_gpl.table.set_index("ID")[disc_sym_col].dropna()
    disc_p2g = disc_p2g[disc_p2g.str.strip() != ""]
    disc_common = disc_expr_df.index.intersection(disc_p2g.index)
    disc_expr_df = disc_expr_df.loc[disc_common]
    disc_expr_df.index = disc_p2g.loc[disc_common].values
    disc_expr_df = disc_expr_df.groupby(disc_expr_df.index).mean()
    disc_gene_expr = disc_expr_df.T
    disc_gene_expr = disc_gene_expr.loc[:, disc_gene_expr.var() > 0]
    
    # Fill NaN in expression data before scoring
    if disc_gene_expr.isna().any().any():
        n_nan = disc_gene_expr.isna().sum().sum()
        disc_gene_expr = disc_gene_expr.fillna(disc_gene_expr.median())
        print(f"Filled {n_nan} NaN values in GSE28521 expression data")
    
    # Subset to frontal cortex
    fc_disc_mask = disc_meta.loc[disc_gene_expr.index, "brain_region"] == "Frontal_Cortex"
    disc_fc_expr = disc_gene_expr.loc[fc_disc_mask]
    disc_fc_meta = disc_meta.loc[fc_disc_mask]
    print(f"GSE28521 frontal cortex: {disc_fc_expr.shape[0]} samples x {disc_fc_expr.shape[1]} genes")
    
    # Score pathways
    disc_scoring = score_pathways_from_expression(
        gene_expression=disc_fc_expr, pathways=pathways,
        method=ExpressionScoringMethod.SSGSEA, min_genes_per_pathway=2, seed=SEED,
    )
    discovery_scores = disc_scoring.pathway_scores
    
    # Handle NaN in pathway scores before clustering
    if discovery_scores.isna().any().any():
        n_nan = discovery_scores.isna().sum().sum()
        print(f"Filling {n_nan} NaN values in pathway scores with column medians")
        discovery_scores = discovery_scores.fillna(discovery_scores.median())
        # Drop any columns still all-NaN
        discovery_scores = discovery_scores.dropna(axis=1, how="all")
    
    print(f"Discovery pathway scores: {discovery_scores.shape[0]} samples x {discovery_scores.shape[1]} pathways")
    
    # Cluster at k=2
    disc_clustering = run_clustering(
        data=discovery_scores.values, n_clusters=2,
        algorithm=ClusteringAlgorithm.GMM, seed=SEED,
    )
    discovery_labels = disc_clustering.labels
    discovery_meta = disc_fc_meta.copy()
    discovery_meta["subtype"] = discovery_labels
    
    print(f"\nRe-ran GSE28521 FC: {discovery_scores.shape[0]} samples, k=2, silhouette={disc_clustering.silhouette:.4f}")
    print(f"Subtypes: {pd.Series(discovery_labels).value_counts().to_dict()}")

print("\nDiscovery cohort (GSE28521 frontal cortex) ready.")

In [ ]:
# Step 2: Train GMM on discovery scores, project validation samples
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score
from scipy.stats import spearmanr

# Align pathway columns between discovery and validation
common_pathways = sorted(set(discovery_scores.columns) & set(fc_scores.columns))
print(f"Common pathways between cohorts: {len(common_pathways)} of {len(discovery_scores.columns)}")

disc_aligned = discovery_scores[common_pathways]
val_aligned = fc_scores[common_pathways]

# Z-score normalize discovery cohort, apply same transform to validation
scaler = StandardScaler()
disc_scaled = scaler.fit_transform(disc_aligned)
val_scaled = scaler.transform(val_aligned)

# Train GMM on discovery data
gmm = GaussianMixture(
    n_components=2,
    covariance_type="full",
    n_init=10,
    reg_covar=1e-6,
    random_state=SEED,
)
gmm.fit(disc_scaled)
disc_predicted = gmm.predict(disc_scaled)
disc_probs = gmm.predict_proba(disc_scaled)

# Project validation cohort
val_predicted = gmm.predict(val_scaled)
val_probs = gmm.predict_proba(val_scaled)

print(f"\nDiscovery GMM trained (k=2)")
print(f"Discovery predicted labels: {pd.Series(disc_predicted).value_counts().to_dict()}")
print(f"Validation projected labels: {pd.Series(val_predicted).value_counts().to_dict()}")

# Map discovery labels to match original subtype numbering
# (GMM label ordering may differ from the original run)
disc_ari = adjusted_rand_score(discovery_labels, disc_predicted)
print(f"\nDiscovery self-ARI (original vs re-trained): {disc_ari:.4f}")

In [ ]:
# Step 3: Analyze projected labels
# Add projected labels to validation metadata
val_meta_proj = fc_meta.copy()
val_meta_proj["projected_subtype"] = val_predicted
val_meta_proj["prob_subtype_0"] = val_probs[:, 0]
val_meta_proj["prob_subtype_1"] = val_probs[:, 1]

# Cross-tab: projected subtypes vs diagnosis
print("=" * 60)
print("CROSS-COHORT PROJECTION: GSE28521 → GSE64018")
print("=" * 60)

ct_proj = pd.crosstab(val_meta_proj["projected_subtype"], val_meta_proj["diagnosis"], margins=True)
print(f"\nProjected subtype × diagnosis:")
print(ct_proj)

# Check ASD purity per projected subtype
print(f"\n--- Projected subtype composition ---")
for subtype in sorted(val_meta_proj["projected_subtype"].unique()):
    mask = val_meta_proj["projected_subtype"] == subtype
    n_total = mask.sum()
    n_asd = (val_meta_proj.loc[mask, "diagnosis"] == "ASD").sum()
    n_ctl = n_total - n_asd
    purity = n_asd / n_total * 100 if n_total > 0 else 0
    print(f"  Projected Subtype {subtype}: {n_asd} ASD + {n_ctl} Control ({purity:.0f}% ASD)")

# Compare independent discovery labels with projected labels
if best_k == 2:
    cross_ari = adjusted_rand_score(best_labels, val_predicted)
    print(f"\nARI between independent discovery and projection: {cross_ari:.4f}")
    print(f"  (>0.3 = moderate agreement, >0.6 = strong agreement)")

In [ ]:
# Step 4: Compare pathway profiles between discovery and validation cohorts

# Identify which subtype in discovery is the "GABA-Collapsed" one
# (the one with lower GABA_SIGNALING score)
disc_subtype_means = discovery_scores.groupby(discovery_labels).mean()
if "GABA_SIGNALING" in disc_subtype_means.columns:
    gaba_col = "GABA_SIGNALING"
else:
    gaba_col = [c for c in disc_subtype_means.columns if 'gaba' in c.lower()]
    gaba_col = gaba_col[0] if gaba_col else disc_subtype_means.columns[0]

disc_gaba_collapsed_id = disc_subtype_means[gaba_col].idxmin()
disc_baseline_id = 1 - disc_gaba_collapsed_id  # assuming k=2

print(f"Discovery GABA-Collapsed subtype: {disc_gaba_collapsed_id}")
print(f"Discovery Baseline subtype: {disc_baseline_id}")

# Mean pathway profiles per subtype in both cohorts
disc_profiles = discovery_scores[common_pathways].groupby(discovery_labels).mean()
val_profiles = fc_scores[common_pathways].groupby(val_predicted).mean()

print(f"\nDiscovery profile subtypes: {list(disc_profiles.index)}")
print(f"Validation projected subtypes: {list(val_profiles.index)}")

# Check which subtypes exist in the validation projection
val_has_both = (disc_gaba_collapsed_id in val_profiles.index and 
                disc_baseline_id in val_profiles.index)

if val_has_both:
    # Both subtypes present — compare profiles directly
    disc_gaba_profile = disc_profiles.loc[disc_gaba_collapsed_id]
    val_gaba_profile = val_profiles.loc[disc_gaba_collapsed_id]
    rho, pval = spearmanr(disc_gaba_profile, val_gaba_profile)
    print(f"\nPathway profile correlation (GABA-Collapsed subtype):")
    print(f"  Spearman rho = {rho:.4f}, p = {pval:.2e}")
    
    disc_base_profile = disc_profiles.loc[disc_baseline_id]
    val_base_profile = val_profiles.loc[disc_baseline_id]
    rho_base, pval_base = spearmanr(disc_base_profile, val_base_profile)
    print(f"\nPathway profile correlation (Baseline subtype):")
    print(f"  Spearman rho = {rho_base:.4f}, p = {pval_base:.2e}")
else:
    # Only one subtype in validation — compare that subtype's profile to both discovery profiles
    present_id = list(val_profiles.index)[0]
    absent_id = disc_gaba_collapsed_id if present_id == disc_baseline_id else disc_baseline_id
    print(f"\nWARNING: All validation samples projected to subtype {present_id}")
    print(f"  Subtype {absent_id} has no validation samples")
    
    val_profile = val_profiles.loc[present_id]
    
    # Compare to both discovery subtypes
    for disc_id, label in [(disc_gaba_collapsed_id, "GABA-Collapsed"), 
                            (disc_baseline_id, "Baseline")]:
        disc_profile = disc_profiles.loc[disc_id]
        r, p = spearmanr(disc_profile, val_profile)
        print(f"\n  Validation (subtype {present_id}) vs Discovery {label} (subtype {disc_id}):")
        print(f"    Spearman rho = {r:.4f}, p = {p:.2e}")
    
    # Set variables for downstream use
    rho, pval = spearmanr(disc_profiles.loc[disc_gaba_collapsed_id], val_profile)
    rho_base, pval_base = spearmanr(disc_profiles.loc[disc_baseline_id], val_profile)

# Visualize pathway profile comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

disc_gaba_profile = disc_profiles.loc[disc_gaba_collapsed_id]
disc_base_profile = disc_profiles.loc[disc_baseline_id]

if val_has_both:
    val_gaba_profile = val_profiles.loc[disc_gaba_collapsed_id]
    val_base_profile = val_profiles.loc[disc_baseline_id]
    left_title = f"GABA-Collapsed (rho={rho:.3f})"
    right_title = f"Baseline (rho={rho_base:.3f})"
else:
    # Show the single validation profile against both discovery profiles
    val_gaba_profile = val_profiles.loc[list(val_profiles.index)[0]]
    val_base_profile = val_gaba_profile  # same profile
    left_title = f"Disc GABA-Collapsed vs Val (rho={rho:.3f})"
    right_title = f"Disc Baseline vs Val (rho={rho_base:.3f})"

x = range(len(common_pathways))

# Left plot
axes[0].barh(x, disc_gaba_profile.values, height=0.4, label="GSE28521 (Discovery)", alpha=0.7, color="coral")
axes[0].barh([i + 0.4 for i in x], val_gaba_profile.values, height=0.4, label="GSE64018 (Validation)", alpha=0.7, color="steelblue")
axes[0].set_yticks([i + 0.2 for i in x])
axes[0].set_yticklabels([p.replace("_", " ") for p in common_pathways], fontsize=8)
axes[0].set_xlabel("Mean Pathway Score")
axes[0].set_title(left_title)
axes[0].legend(fontsize=9)
axes[0].axvline(x=0, color="black", linewidth=0.5)

# Right plot
axes[1].barh(x, disc_base_profile.values, height=0.4, label="GSE28521 (Discovery)", alpha=0.7, color="coral")
axes[1].barh([i + 0.4 for i in x], val_base_profile.values, height=0.4, label="GSE64018 (Validation)", alpha=0.7, color="steelblue")
axes[1].set_yticks([i + 0.2 for i in x])
axes[1].set_yticklabels([p.replace("_", " ") for p in common_pathways], fontsize=8)
axes[1].set_xlabel("Mean Pathway Score")
axes[1].set_title(right_title)
axes[1].legend(fontsize=9)
axes[1].axvline(x=0, color="black", linewidth=0.5)

plt.suptitle("Cross-Cohort Pathway Profile Comparison: GSE28521 vs GSE64018",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(FC_OUTPUT_DIR, "gse64018_cross_cohort_profiles.png"), dpi=150, bbox_inches="tight")
plt.show()

## 8. Summary & Export

In [ ]:
# Export characterization CSVs
export_characterization(best_char, output_dir=FC_OUTPUT_DIR, formats=["csv"])

# Save metadata with subtypes
best_meta.to_csv(os.path.join(FC_OUTPUT_DIR, "gse64018_fc_sample_metadata.csv"))
fc_scores.to_csv(os.path.join(FC_OUTPUT_DIR, "gse64018_fc_pathway_scores.csv"))

# Save projection results
val_meta_proj.to_csv(os.path.join(FC_OUTPUT_DIR, "gse64018_fc_projected_subtypes.csv"))

# Safely get cross_ari (may not be defined if best_k != 2)
try:
    cross_ari_val = float(cross_ari)
except NameError:
    cross_ari_val = None

# Save comprehensive results JSON
results_summary = {
    "analysis": "cross_cohort_validation",
    "discovery_dataset": "GSE28521",
    "discovery_citation": "Voineagu et al. 2011, Nature",
    "discovery_platform": "Illumina HumanRef-8 v3.0 (microarray)",
    "validation_dataset": "GSE64018",
    "validation_citation": "Gupta et al. 2014, Nature Communications",
    "validation_platform": "Illumina HiSeq 2000 (RNA-seq)",
    "validation_region": str(target_region) if target_region else "all",
    "validation_n_samples": int(len(fc_scores)),
    "validation_n_asd": int((fc_meta["diagnosis"] == "ASD").sum()),
    "validation_n_control": int((fc_meta["diagnosis"] == "Control").sum()),
    "independent_discovery": {
        "best_k": int(best_k),
        "silhouette": float(fc_all_results[best_k]["silhouette"]),
        "n_gates_passed": int(fc_all_results[best_k]["n_gates_passed"]),
        "benchmark_winner": str(fc_bench.best_method),
        "results_by_k": {
            str(k): {
                "silhouette": float(fc_all_results[k]["silhouette"]),
                "n_gates_passed": int(fc_all_results[k]["n_gates_passed"]),
                "validation_gates": [
                    {"name": str(g.name), "passed": bool(g.passed),
                     "metric": str(g.metric_name), "value": float(g.metric_value),
                     "threshold": float(g.threshold)}
                    for g in fc_all_results[k]["validation"].results
                ],
            }
            for k in [2, 3, 4]
        },
    },
    "cross_cohort_projection": {
        "discovery_self_ari": float(disc_ari),
        "independent_vs_projected_ari": cross_ari_val,
        "projection_both_subtypes_present": val_has_both,
        "gaba_collapsed_profile_correlation": {
            "spearman_rho": float(rho),
            "p_value": float(pval),
        },
        "baseline_profile_correlation": {
            "spearman_rho": float(rho_base),
            "p_value": float(pval_base),
        },
        "projected_subtype_composition": {
            str(subtype): {
                "n_asd": int((val_meta_proj.loc[val_meta_proj["projected_subtype"] == subtype, "diagnosis"] == "ASD").sum()),
                "n_control": int((val_meta_proj.loc[val_meta_proj["projected_subtype"] == subtype, "diagnosis"] == "Control").sum()),
            }
            for subtype in sorted(val_meta_proj["projected_subtype"].unique())
        },
    },
    "framework_version": "0.3.0",
    "seed": SEED,
}

with open(os.path.join(FC_OUTPUT_DIR, "gse64018_results_summary.json"), "w") as f:
    json.dump(results_summary, f, indent=2)

print("\n" + "=" * 60)
print("CROSS-COHORT VALIDATION COMPLETE")
print("=" * 60)
print(f"\nDiscovery: GSE28521 (Voineagu et al. 2011, microarray)")
print(f"Validation: GSE64018 (Gupta et al. 2014, RNA-seq)")
print(f"\n--- Independent Discovery on GSE64018 ---")
print(f"Region: {target_region}")
print(f"Samples: {len(fc_scores)} ({(fc_meta['diagnosis'] == 'ASD').sum()} ASD, {(fc_meta['diagnosis'] == 'Control').sum()} Control)")
print(f"Best k: {best_k}, Silhouette: {fc_all_results[best_k]['silhouette']:.4f}")
print(f"Benchmark winner: {fc_bench.best_method}")
print(f"\n--- Cross-Cohort Projection ---")
print(f"Both subtypes present in projection: {val_has_both}")
print(f"GABA-Collapsed profile correlation: rho={rho:.4f} (p={pval:.2e})")
print(f"Baseline profile correlation: rho={rho_base:.4f} (p={pval_base:.2e})")
if cross_ari_val is not None:
    print(f"Independent vs projected ARI: {cross_ari_val:.4f}")
print(f"\nOutputs saved to: {FC_OUTPUT_DIR}/")

---

## References

1. Gupta S, et al. (2014). Transcriptome analysis reveals dysregulation of innate immune response genes and neuronal activity-dependent genes in autism. *Nature Communications*, 5:5748. [PMID: 25494366](https://pubmed.ncbi.nlm.nih.gov/25494366/)
2. Voineagu I, et al. (2011). Transcriptomic analysis of autistic brain reveals convergent molecular pathology. *Nature*, 474(7351):380-384. [PMID: 21614001](https://pubmed.ncbi.nlm.nih.gov/21614001/)
3. Chauhan R (2026). Pathway Subtyping Framework v0.3.0. *Zenodo*. [DOI: 10.5281/zenodo.18442426](https://doi.org/10.5281/zenodo.18442426)

## Data Availability

- **GSE64018:** https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE64018
- **GSE28521:** https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE28521
- **Framework:** https://github.com/topmist-admin/pathway-subtyping-framework
- **PyPI:** `pip install pathway-subtyping`

## License

This notebook is released under CC-BY 4.0.